# Homework 6 - Decision Trees and Ensemble Learning

ML Zoomcamp 2026

## Dataset

We continue using the fuel efficiency dataset (`car_fuel_efficiency_2026.csv`).
The goal is to predict `fuel_efficiency_mpg`.

## Preparing the dataset

* Fill missing values with zeros.
* Split with `test_size=0.2, random_state=1` then `test_size=0.25, random_state=1`.
* Use `DictVectorizer(sparse=True)`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv("car_fuel_efficiency_2026.csv").fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train["fuel_efficiency_mpg"], df_val["fuel_efficiency_mpg"], df_test["fuel_efficiency_mpg"]

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient="records"))
X_val = dv.transform(df_val.to_dict(orient="records"))
X_test = dv.transform(df_test.to_dict(orient="records"))
names = dv.get_feature_names_out()
print("X_train:", X_train.shape, type(X_train).__name__)
print("split sizes:", len(df_train), len(df_val), len(df_test))


def rmse(a, b):
    return float(np.sqrt(((a - b) ** 2).mean()))

X_train: (6000, 16) csr_matrix
split sizes: 6000 2000 2000


## Question 1

Train a decision tree regressor with `max_depth=1`. Which original feature is used for splitting the data? If you use `DictVectorizer`, map a one-hot feature such as `origin=USA` back to its original column name.

* `vehicle_weight`
* `model_year`
* `origin`
* `fuel_type`

In [2]:
print("\n===== Q1 : DecisionTreeRegressor(max_depth=1) =====")
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)
feature = names[dt.tree_.feature[0]]
print("split feature :", feature)
print("original column:", feature.split("=")[0])


===== Q1 : DecisionTreeRegressor(max_depth=1) =====
split feature : model_year
original column: model_year


## Question 2

Train a random forest regressor with `n_estimators=10`, `random_state=1`, `n_jobs=-1`. What's the RMSE on the validation data?

* 0.1837
* 1.837
* 18.37
* 183.7

In [3]:
print("\n===== Q2 : RandomForestRegressor(n_estimators=10) =====")
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
q2 = rmse(y_val, rf.predict(X_val))
print("val RMSE =", q2, "->", round(q2, 4))


===== Q2 : RandomForestRegressor(n_estimators=10) =====


val RMSE = 1.8332197358745623 -> 1.8332


## Question 3

Try `n_estimators` in `[10, 50, 100, 150]` with `random_state=1`. Which value gives the lowest validation RMSE? (consider 3 decimals)

In [4]:
print("\n===== Q3 : n_estimators sweep =====")
q3 = {}
for n in [10, 50, 100, 150]:
    m = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    m.fit(X_train, y_train)
    r = rmse(y_val, m.predict(X_val))
    q3[n] = round(r, 3)
    print(f"  n_estimators={n:3d}  RMSE={r:.6f}  round3={round(r, 3)}")
print("lowest ->", min(q3, key=q3.get))


===== Q3 : n_estimators sweep =====


  n_estimators= 10  RMSE=1.833220  round3=1.833


  n_estimators= 50  RMSE=1.773988  round3=1.774


  n_estimators=100  RMSE=1.769786  round3=1.77


  n_estimators=150  RMSE=1.769198  round3=1.769
lowest -> 150


## Question 4

Select the best `max_depth` from `[10, 15, 20, 25]`. For each value, try `n_estimators` from `[10, 50, 100, 150]` and calculate the mean RMSE. `random_state=1`. What's the best `max_depth`?

In [5]:
print("\n===== Q4 : max_depth x n_estimators =====")
q4 = {}
for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        m = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        m.fit(X_train, y_train)
        scores.append(rmse(y_val, m.predict(X_val)))
    q4[d] = float(np.mean(scores))
    print(f"  max_depth={d}: {[round(s, 6) for s in scores]}  mean={q4[d]:.6f}")
print("best max_depth ->", min(q4, key=q4.get))


===== Q4 : max_depth x n_estimators =====


  max_depth=10: [1.782348, 1.74752, 1.743986, 1.743716]  mean=1.754392


  max_depth=15: [1.83139, 1.770883, 1.767872, 1.766752]  mean=1.784224


  max_depth=20: [1.837314, 1.772377, 1.768918, 1.768476]  mean=1.786771


  max_depth=25: [1.836852, 1.772123, 1.76832, 1.767927]  mean=1.786305
best max_depth -> 10


## Question 5

Train a random forest with `n_estimators=10`, `max_depth=20`, `random_state=1` and get `feature_importances_`. What's the most important feature among these 4?

* `vehicle_weight`
* `horsepower`
* `acceleration`
* `engine_displacement`

In [6]:
print("\n===== Q5 : feature importance (n_estimators=10, max_depth=20) =====")
m5 = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
m5.fit(X_train, y_train)
imp = sorted(zip(names, m5.feature_importances_), key=lambda t: -t[1])
for name, score in imp[:8]:
    print(f"  {name:40s} {score:.6f}")


===== Q5 : feature importance (n_estimators=10, max_depth=20) =====


  model_year                               0.296107
  vehicle_weight                           0.203313
  fuel_type=Gasoline                       0.160712
  horsepower                               0.078271
  engine_displacement                      0.061281
  acceleration                             0.054998
  origin=USA                               0.042152
  drivetrain=Front-wheel drive             0.020062


## Question 6

Train an XGBoost model and tune `eta`. Create DMatrix for train and validation, a watchlist, and train for 100 rounds with the given params. Compare `eta=0.3` vs `eta=0.1`. Which eta gives the best validation RMSE?

In [7]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(names))
dval = xgb.DMatrix(X_val, label=y_val, feature_names=list(names))
watchlist = [(dtrain, "train"), (dval, "val")]

print("xgboost", xgb.__version__)

for eta in [0.3, 0.1]:
    params = {
        "eta": eta,
        "max_depth": 6,
        "min_child_weight": 1,
        "objective": "reg:squarederror",
        "nthread": 8,
        "seed": 1,
        "verbosity": 1,
    }
    model = xgb.train(params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    pred = model.predict(dval)
    score = float(np.sqrt(((y_val - pred) ** 2).mean()))
    print(f"eta={eta}  val RMSE = {score:.6f}  -> {round(score, 4)}")

xgboost 3.4.1


eta=0.3  val RMSE = 1.826458  -> 1.8265


eta=0.1  val RMSE = 1.724810  -> 1.7248
